[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/a-ramakrishnan/vision_transformers/blob/main/autoencoder_vae/autoencoder_vae.ipynb)

In [1]:
import os
import sys
import subprocess

# Default Colab GPU preference: A100 (fallback: L4)
if "google.colab" in sys.modules and not os.path.exists("models.py"):
    if not os.path.exists("vision_transformers"):
        subprocess.run(
            ["git", "clone", "https://github.com/a-ramakrishnan/vision_transformers.git"],
            check=True,
        )
    os.chdir("vision_transformers")
    if not os.path.exists("autoencoder_vae/models.py"):
        subprocess.run(["git", "fetch", "origin"], check=True)
        subprocess.run(["git", "checkout", "feat/modular-structure"], check=True)
    os.chdir("autoencoder_vae")

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

In [2]:
import torch
from dataset import get_mnist_dataloaders
from models import Autoencoder, VAE
from train import train_autoencoder, train_vae, save_training_results
from visualize import plot_reconstructions, plot_vae_samples

device = torch.device(
    "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
)
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else str(device)
print(f"Using device: {device} ({gpu_name})")

BATCH_SIZE = 128
train_loader, test_loader = get_mnist_dataloaders(batch_size=BATCH_SIZE)

Using device: cuda (NVIDIA L4)


In [3]:
# =====================================================================
# PART 1: Train & Visualize Vanilla Autoencoder
# =====================================================================
AE_EPOCHS = 10
AE_LR = 1e-3
AE_LATENT_DIM = 2

ae_model = Autoencoder(latent_dim=AE_LATENT_DIM).to(device)
ae_model, ae_history = train_autoencoder(
    ae_model,
    train_loader,
    epochs=AE_EPOCHS,
    lr=AE_LR,
    device=device,
    checkpoint_path="autoencoder_mnist.pth",
)

plot_reconstructions(
    ae_model,
    test_loader,
    device=device,
    is_vae=False,
    title="Vanilla Autoencoder: Original (Top) vs Reconstructed (Bottom)",
    save_path="ae_reconstructions.png",
)

[Autoencoder] Epoch [1/10] - MSE Loss: 0.061032
[Autoencoder] Epoch [2/10] - MSE Loss: 0.047800
[Autoencoder] Epoch [3/10] - MSE Loss: 0.044566
[Autoencoder] Epoch [4/10] - MSE Loss: 0.043063
[Autoencoder] Epoch [5/10] - MSE Loss: 0.042065
[Autoencoder] Epoch [6/10] - MSE Loss: 0.041254
[Autoencoder] Epoch [7/10] - MSE Loss: 0.040655
[Autoencoder] Epoch [8/10] - MSE Loss: 0.040100
[Autoencoder] Epoch [9/10] - MSE Loss: 0.039696
[Autoencoder] Epoch [10/10] - MSE Loss: 0.039341


In [4]:
# =====================================================================
# PART 2: Train & Visualize Variational Autoencoder (VAE)
# =====================================================================
VAE_EPOCHS = 15
VAE_LR = 1e-3
VAE_LATENT_DIM = 20

vae_model = VAE(input_dim=784, hidden_dim=400, latent_dim=VAE_LATENT_DIM).to(device)
vae_model, vae_history = train_vae(
    vae_model,
    train_loader,
    epochs=VAE_EPOCHS,
    lr=VAE_LR,
    device=device,
    checkpoint_path="vae_mnist.pth",
)

save_training_results(ae_history, vae_history, save_path="training_results.json")

plot_reconstructions(
    vae_model,
    test_loader,
    device=device,
    is_vae=True,
    title="VAE: Original (Top) vs Reconstructed (Bottom)",
    save_path="vae_reconstructions.png",
)
plot_vae_samples(
    vae_model,
    latent_dim=VAE_LATENT_DIM,
    num_samples=16,
    device=device,
    save_path="vae_samples.png",
)

[VAE] Epoch [1/15] - Total Loss: 165.1622 | BCE: 149.8300 | KLD: 15.3322
[VAE] Epoch [2/15] - Total Loss: 120.9769 | BCE: 98.5390 | KLD: 22.4379
[VAE] Epoch [3/15] - Total Loss: 114.2920 | BCE: 90.3139 | KLD: 23.9781
[VAE] Epoch [4/15] - Total Loss: 111.3720 | BCE: 86.8196 | KLD: 24.5525
[VAE] Epoch [5/15] - Total Loss: 109.6193 | BCE: 84.8147 | KLD: 24.8046
[VAE] Epoch [6/15] - Total Loss: 108.5424 | BCE: 83.5763 | KLD: 24.9661
[VAE] Epoch [7/15] - Total Loss: 107.6935 | BCE: 82.6543 | KLD: 25.0392
[VAE] Epoch [8/15] - Total Loss: 107.0712 | BCE: 81.9524 | KLD: 25.1188
[VAE] Epoch [9/15] - Total Loss: 106.6449 | BCE: 81.4487 | KLD: 25.1961
[VAE] Epoch [10/15] - Total Loss: 106.1455 | BCE: 80.9085 | KLD: 25.2370
[VAE] Epoch [11/15] - Total Loss: 105.8656 | BCE: 80.5740 | KLD: 25.2915
[VAE] Epoch [12/15] - Total Loss: 105.5318 | BCE: 80.2274 | KLD: 25.3044
[VAE] Epoch [13/15] - Total Loss: 105.2104 | BCE: 79.9125 | KLD: 25.2979
[VAE] Epoch [14/15] - Total Loss: 104.9812 | BCE: 79.6821 |